In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import itertools
import networkx.algorithms.community as nx_comm
import random
from collections import defaultdict
import pickle
from collections import Counter

In [2]:
with open('data/processed/all_c_elegans_weighted_graphs.pkl', 'rb') as file:
    all_c_elegans_weighted_graphs = pickle.load(file)

In [3]:
def consensus_community_detection(graph, number_of_iterations):
    
    random_seeds = random.sample(range(1, 100001), number_of_iterations)
    dict_100_iterations_of_communitydetection = {}
    for i in range(number_of_iterations):
        dict_100_iterations_of_communitydetection[i] = nx_comm.louvain_communities(graph, weight='weight', seed=random_seeds[i])
    
    node_map = {node: i for i, node in enumerate(sorted(graph.nodes()))}
        
    n = len(graph.nodes)
    
    # Step 1: Initialize the matrix M
    M = np.zeros((n, n))

    # Step 2 and 3: Loop through the iterations of community detection
    for list_of_communities in dict_100_iterations_of_communitydetection.values():
        B = np.zeros((n, len(list_of_communities))) 
        for i, community in enumerate(list_of_communities):
            for node in community:
                B[node_map[node], i] = 1 # B[i,j] = 1 if node i is in the jth community
        C = np.dot(B, B.T) # C[i, j] - no. of times node i & node j are in the same community for this particular iteration.

        # Step 4: Add C to M
        M += C
        
    
    # creating an undirected, weighted graph from consensus matrix 
    G_after_100_louvain = nx.from_numpy_array(M)
    
    # Remember: the node ids in M and therefore G_after_100_louvain had to be made continuous to execute matrix operations
    
    # removing self edges from consensus graph; I tried with and without it: it does alter the final community sets :O 
    G_after_100_louvain.remove_edges_from(nx.selfloop_edges(G_after_100_louvain)) 
    
    consensus_community = nx_comm.louvain_communities(G_after_100_louvain, weight='weight') # considering edge weights does give slightly different results

    node_map_reversed = {v:k for k,v in node_map.items()}

    consensus_community_reverse_renumbering = []
    for community in consensus_community:
        a_set = set()
        for node in community:
            a_set.add(node_map_reversed[node])
        consensus_community_reverse_renumbering.append(a_set)
    
    consensus_community_reverse_renumbering = sorted(consensus_community_reverse_renumbering, key=len, reverse=False)

    list_of_nodes_in_order_of_communities = [val for sublist in consensus_community_reverse_renumbering for val in sublist]

    dict_of_nodes_in_order_of_communities_renumbered = {}
    for index, MyNodeId  in enumerate(list_of_nodes_in_order_of_communities):
        dict_of_nodes_in_order_of_communities_renumbered[MyNodeId] = index
    
    consensus_community_renumbered = []
    for community in consensus_community_reverse_renumbering:
        newcommunity = set()
        for node in community:
            newcommunity.add(dict_of_nodes_in_order_of_communities_renumbered[node])
        consensus_community_renumbered.append(newcommunity)
    
    # returning communities with original ids, with renumbered ids for community scatterplot, consensus matrix
    return consensus_community_reverse_renumbering, consensus_community_renumbered, M

In [24]:
CCD_of_all_weighted_graphs = {}
for i, network in enumerate(all_c_elegans_weighted_graphs):
    CCD_of_all_weighted_graphs[i] = consensus_community_detection(network, 100)

In [11]:
#with open('data/processed/C_elegans_CCD_of_all_weighted_graphs.pkl', 'wb') as file:
    #pickle.dump(CCD_of_all_weighted_graphs, file)